In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

print(DATA_DIR)

..\data\raw


In [2]:
files = list(DATA_DIR.glob("*.csv"))

print(f"Number of datasets: {len(files)}")

for file in files:
    print(file.name)

Number of datasets: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
datasets = {}

for file in files:
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    datasets[name] = pd.read_csv(file)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [4]:
for name, df in datasets.items():
    print(f"{name:35} {df.shape}")

customers                           (99441, 5)
geolocation                         (1000163, 5)
orders                              (99441, 8)
order_items                         (112650, 7)
order_payments                      (103886, 5)
order_reviews                       (99224, 7)
products                            (32951, 9)
sellers                             (3095, 4)
product_category_name_translation   (71, 2)


In [5]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.columns.tolist())


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

ORDER_REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_heigh

In [6]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.dtypes)


CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

ORDER_PAYMENTS
order

In [7]:
data_dictionary = []

for name, df in datasets.items():
    for column in df.columns:
        data_dictionary.append({
            "table": name,
            "column": column,
            "dtype": str(df[column].dtype),
            "missing": df[column].isna().sum(),
            "unique_values": df[column].nunique()
        })

data_dictionary = pd.DataFrame(data_dictionary)

data_dictionary

,table,column,dtype,missing,unique_values
0,customers,customer_id,str,0,99441
1,customers,customer_unique_id,str,0,96096
2,customers,customer_zip_code_prefix,int64,0,14994
3,customers,customer_city,str,0,4119
4,customers,customer_state,str,0,27
5,geolocation,geolocation_zip_code_prefix,int64,0,19015
6,geolocation,geolocation_lat,float64,0,717360
7,geolocation,geolocation_lng,float64,0,717613
8,geolocation,geolocation_city,str,0,8011
9,geolocation,geolocation_state,str,0,27


In [8]:
data_dictionary.to_csv(
    "../reports/data_dictionary.csv",
    index=False
)

print("Data dictionary saved successfully.")

Data dictionary saved successfully.


In [9]:
for name, df in datasets.items():
    id_columns = [
        column for column in df.columns
        if column.endswith("_id")
    ]

    print(f"\n{name.upper()}")
    print(id_columns)


CUSTOMERS
['customer_id', 'customer_unique_id']

GEOLOCATION
[]

ORDERS
['order_id', 'customer_id']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id']

ORDER_PAYMENTS
['order_id']

ORDER_REVIEWS
['review_id', 'order_id']

PRODUCTS
['product_id']

SELLERS
['seller_id']

PRODUCT_CATEGORY_NAME_TRANSLATION
[]


In [10]:
key_checks = {
    "customers.customer_id": datasets["customers"]["customer_id"],
    "customers.customer_unique_id": datasets["customers"]["customer_unique_id"],
    "orders.order_id": datasets["orders"]["order_id"],
    "products.product_id": datasets["products"]["product_id"],
    "sellers.seller_id": datasets["sellers"]["seller_id"],
    "order_reviews.review_id": datasets["order_reviews"]["review_id"],
    "order_items.order_id": datasets["order_items"]["order_id"],
    "order_items.product_id": datasets["order_items"]["product_id"],
    "order_items.seller_id": datasets["order_items"]["seller_id"],
}

for name, series in key_checks.items():
    print(
        f"{name:40} "
        f"rows={len(series):,} | "
        f"unique={series.nunique():,} | "
        f"missing={series.isna().sum():,}"
    )

customers.customer_id                    rows=99,441 | unique=99,441 | missing=0
customers.customer_unique_id             rows=99,441 | unique=96,096 | missing=0
orders.order_id                          rows=99,441 | unique=99,441 | missing=0
products.product_id                      rows=32,951 | unique=32,951 | missing=0
sellers.seller_id                        rows=3,095 | unique=3,095 | missing=0
order_reviews.review_id                  rows=99,224 | unique=98,410 | missing=0
order_items.order_id                     rows=112,650 | unique=98,666 | missing=0
order_items.product_id                   rows=112,650 | unique=32,951 | missing=0
order_items.seller_id                    rows=112,650 | unique=3,095 | missing=0


In [11]:
order_items = datasets["order_items"]

composite_key = (
    order_items["order_id"].astype(str)
    + "_"
    + order_items["order_item_id"].astype(str)
)

print(f"Total order items: {len(composite_key):,}")
print(f"Unique order/item combinations: {composite_key.nunique():,}")
print(f"Duplicate combinations: {composite_key.duplicated().sum():,}")

Total order items: 112,650
Unique order/item combinations: 112,650
Duplicate combinations: 0


In [12]:
missing_summary = []

for name, df in datasets.items():
    total_missing = df.isna().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    missing_pct = (total_missing / total_cells) * 100

    missing_summary.append({
        "table": name,
        "missing_values": total_missing,
        "missing_percentage": round(missing_pct, 2)
    })

missing_summary = pd.DataFrame(missing_summary)

missing_summary.sort_values(
    "missing_values",
    ascending=False
)

,table,missing_values,missing_percentage
5,order_reviews,145903,21.01
2,orders,4908,0.62
6,products,2448,0.83
0,customers,0,0.00
1,geolocation,0,0.00
4,order_payments,0,0.00
3,order_items,0,0.00
7,sellers,0,0.00
8,product_category_name_translation,0,0.00


In [13]:
missing_details = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_details.append({
                "table": name,
                "column": column,
                "missing_values": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100,
                    2
                )
            })

missing_details = pd.DataFrame(missing_details)

missing_details.sort_values(
    "missing_values",
    ascending=False
)

,table,column,missing_values,missing_percentage
3,order_reviews,review_comment_title,87656,88.34
4,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
6,products,product_name_lenght,610,1.85
8,products,product_photos_qty,610,1.85
0,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [14]:
missing_details = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_details.append({
                "table": name,
                "column": column,
                "missing_values": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100,
                    2
                )
            })

missing_details = pd.DataFrame(missing_details)

missing_details.sort_values(
    "missing_values",
    ascending=False
)

,table,column,missing_values,missing_percentage
3,order_reviews,review_comment_title,87656,88.34
4,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
6,products,product_name_lenght,610,1.85
8,products,product_photos_qty,610,1.85
0,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [15]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "table": name,
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            duplicate_count / len(df) * 100,
            2
        )
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary.sort_values(
    "duplicate_rows",
    ascending=False
)

,table,duplicate_rows,duplicate_percentage
1,geolocation,261831,26.18
0,customers,0,0.00
2,orders,0,0.00
3,order_items,0,0.00
4,order_payments,0,0.00
5,order_reviews,0,0.00
6,products,0,0.00
7,sellers,0,0.00
8,product_category_name_translation,0,0.00


In [16]:
geo = datasets["geolocation"]

print("Total rows:", len(geo))
print("Unique rows:", len(geo.drop_duplicates()))
print("Duplicate rows:", geo.duplicated().sum())

Total rows: 1000163
Unique rows: 738332
Duplicate rows: 261831


In [17]:
geo[geo["geolocation_zip_code_prefix"] == 1000].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


In [18]:
print(
    "Unique ZIP-code prefixes:",
    geo["geolocation_zip_code_prefix"].nunique()
)

Unique ZIP-code prefixes: 19015


In [19]:
zip_counts = (
    geo.groupby("geolocation_zip_code_prefix")
       .size()
       .sort_values(ascending=False)
)

zip_counts.describe()

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64

In [20]:
example_zip = geo["geolocation_zip_code_prefix"].iloc[0]

print("Example ZIP prefix:", example_zip)

geo[
    geo["geolocation_zip_code_prefix"] == example_zip
].head(10)

Example ZIP prefix: 1037


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
14,1037,-23.545187,-46.637855,são paulo,SP
31,1037,-23.546705,-46.640336,são paulo,SP
169,1037,-23.543883,-46.638075,são paulo,SP
178,1037,-23.546157,-46.639885,sao paulo,SP
236,1037,-23.543883,-46.638075,sao paulo,SP
283,1037,-23.545199,-46.637916,sao paulo,SP
417,1037,-23.545187,-46.637855,sao paulo,SP
420,1037,-23.546723,-46.640281,sao paulo,SP
433,1037,-23.546463,-46.640145,sao paulo,SP


In [21]:
geo_clean = geo.drop_duplicates().copy()

In [22]:
zip_coordinate_counts = (
    geo.groupby("geolocation_zip_code_prefix")
       .agg(
           unique_coordinates=(
               "geolocation_lat",
               "nunique"
           ),
           rows=("geolocation_zip_code_prefix", "size")
       )
       .sort_values("unique_coordinates", ascending=False)
)

zip_coordinate_counts.head(10)

,unique_coordinates,rows
geolocation_zip_code_prefix,,
38400,746,965
11680,726,879
35500,726,907
11740,666,788
36400,627,733
39400,618,724
35162,611,713
38408,600,773
37200,595,696


In [24]:
orders = datasets["orders"]

orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [25]:
missing_delivery_by_status = (
    orders[orders["order_delivered_customer_date"].isna()]
    ["order_status"]
    .value_counts()
)

missing_delivery_by_status

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [26]:
status_analysis = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_delivery_date=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
)

status_analysis["missing_delivery_pct"] = (
    status_analysis["missing_delivery_date"]
    / status_analysis["total_orders"]
    * 100
)

status_analysis.sort_values(
    "missing_delivery_date",
    ascending=False
)

,total_orders,missing_delivery_date,missing_delivery_pct
order_status,,,
shipped,1107,1107,100.000000
canceled,625,619,99.040000
unavailable,609,609,100.000000
invoiced,314,314,100.000000
processing,301,301,100.000000
delivered,96478,8,0.008292
created,5,5,100.000000
approved,2,2,100.000000


In [29]:
carrier_missing_by_status = (
    orders[orders["order_delivered_carrier_date"].isna()]
    ["order_status"]
    .value_counts()
)

carrier_missing_by_status

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [31]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [32]:
carrier_missing_by_status

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [33]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [34]:
invalid_purchase_approval = orders[
    orders["order_approved_at"].notna() &
    (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    )
]

print(
    "Orders approved before purchase:",
    len(invalid_purchase_approval)
)

Orders approved before purchase: 0


In [35]:
invalid_approval_carrier = orders[
    orders["order_delivered_carrier_date"].notna() &
    orders["order_approved_at"].notna() &
    (
        orders["order_delivered_carrier_date"]
        < orders["order_approved_at"]
    )
]

print(
    "Orders sent to carrier before approval:",
    len(invalid_approval_carrier)
)

Orders sent to carrier before approval: 1359


In [36]:
invalid_approval_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19


In [37]:
invalid_approval_carrier = invalid_approval_carrier.copy()

invalid_approval_carrier["time_difference_hours"] = (
    invalid_approval_carrier["order_approved_at"]
    - invalid_approval_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

invalid_approval_carrier["time_difference_hours"].describe()

count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: time_difference_hours, dtype: float64

In [38]:
inconsistency_pct = (
    len(invalid_approval_carrier)
    / len(orders)
    * 100
)

print(
    f"Orders with approval/carrier inconsistency: "
    f"{len(invalid_approval_carrier):,}"
)

print(
    f"Percentage of all orders: "
    f"{inconsistency_pct:.2f}%"
)

Orders with approval/carrier inconsistency: 1,359
Percentage of all orders: 1.37%


In [39]:
invalid_approval_carrier["order_status"].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [41]:
inconsistency_pct = (
    len(invalid_approval_carrier)
    / len(orders)
    * 100
)

print(
    f"Orders with approval/carrier inconsistency: "
    f"{len(invalid_approval_carrier):,}"
)

print(
    f"Percentage of all orders: "
    f"{inconsistency_pct:.2f}%"
)

Orders with approval/carrier inconsistency: 1,359
Percentage of all orders: 1.37%


In [42]:
invalid_approval_carrier["order_status"].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [43]:
invalid_carrier_customer = orders[
    orders["order_delivered_carrier_date"].notna() &
    orders["order_delivered_customer_date"].notna() &
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    )
]

print(
    "Customer delivery before carrier delivery:",
    len(invalid_carrier_customer)
)

Customer delivery before carrier delivery: 23


In [45]:
invalid_carrier_customer[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [47]:
invalid_carrier_customer = invalid_carrier_customer.copy()

invalid_carrier_customer["time_difference_hours"] = (
    invalid_carrier_customer["order_delivered_carrier_date"]
    - invalid_carrier_customer["order_delivered_customer_date"]
).dt.total_seconds() / 3600

invalid_carrier_customer["time_difference_hours"].describe()

count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: time_difference_hours, dtype: float64

In [48]:
late_deliveries = orders[
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
    & (
        orders["order_delivered_customer_date"]
        > orders["order_estimated_delivery_date"]
    )
]

print("Late deliveries:", len(late_deliveries))
print(
    "Late delivery percentage:",
    round(len(late_deliveries) / len(orders) * 100, 2),
)

Late deliveries: 7827
Late delivery percentage: 7.87


In [49]:
clean_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

print("Clean datasets created successfully.")

for name, df in clean_datasets.items():
    print(f"{name:40} {df.shape}")

Clean datasets created successfully.
customers                                (99441, 5)
geolocation                              (1000163, 5)
orders                                   (99441, 8)
order_items                              (112650, 7)
order_payments                           (103886, 5)
order_reviews                            (99224, 7)
products                                 (32951, 9)
sellers                                  (3095, 4)
product_category_name_translation        (71, 2)


In [50]:
geo_clean = clean_datasets["geolocation"].drop_duplicates().copy()

print("Original rows:", len(clean_datasets["geolocation"]))
print("Clean rows:", len(geo_clean))
print("Removed duplicates:", len(clean_datasets["geolocation"]) - len(geo_clean))

Original rows: 1000163
Clean rows: 738332
Removed duplicates: 261831


In [51]:
clean_datasets["geolocation"] = geo_clean

print("Geolocation cleaned successfully.")
print("New shape:", clean_datasets["geolocation"].shape)

Geolocation cleaned successfully.
New shape: (738332, 5)


In [52]:
print(
    "Remaining duplicate rows:",
    clean_datasets["geolocation"].duplicated().sum()
)

Remaining duplicate rows: 0


In [53]:
products_clean = clean_datasets["products"].copy()

missing_product_info = products_clean[
    products_clean["product_category_name"].isna()
]

print("Products with missing category:",
      len(missing_product_info))

missing_product_info.head(10)

Products with missing category: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [54]:
product_missing_pattern = (
    products_clean.isna()
    .groupby([
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ])
    .size()
    .sort_values(ascending=False)
)

product_missing_pattern

product_category_name  product_name_lenght  product_description_lenght  product_photos_qty
False                  False                False                       False                 32341
True                   True                 True                        True                    610
dtype: int64

In [55]:
physical_missing = products_clean[
    products_clean[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

print("Products with missing physical information:",
      len(physical_missing))

physical_missing

Products with missing physical information: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
order_items_clean = clean_datasets["order_items"]

physical_missing_ids = physical_missing["product_id"]

for product_id in physical_missing_ids:
    count = (
        order_items_clean["product_id"] == product_id
    ).sum()

    print(
        f"{product_id}: {count} order item(s)"
    )

09ff539a621711667c43eba6a3bd8466: 1 order item(s)
5eb564652db742ff8f28759cd8d2652a: 17 order item(s)


In [57]:
payments_clean = clean_datasets["order_payments"]

payment_counts = (
    payments_clean
    .groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

payment_counts

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [58]:
payment_duplicates = payments_clean.duplicated().sum()

print(f"Exact duplicate payment rows: {payment_duplicates}")

Exact duplicate payment rows: 0


In [59]:
payment_value_summary = payments_clean["payment_value"].describe()

payment_value_summary

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [60]:
zero_payments = payments_clean[
    payments_clean["payment_value"] == 0
]

print(f"Zero-payment rows: {len(zero_payments)}")
zero_payments

Zero-payment rows: 9


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [61]:
payment_type_counts = (
    payments_clean["payment_type"]
    .value_counts(dropna=False)
)

payment_type_counts

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [62]:
reviews_clean = clean_datasets["order_reviews"]

review_score_counts = (
    reviews_clean["review_score"]
    .value_counts()
    .sort_index()
)

review_score_counts

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [63]:
review_comment_analysis = (
    reviews_clean
    .assign(
        has_comment=reviews_clean["review_comment_message"].notna()
    )
    .groupby("review_score")["has_comment"]
    .agg(["count", "sum", "mean"])
)

review_comment_analysis

,count,sum,mean
review_score,,,
1,11424,8745,0.765494
2,3151,2145,0.680736
3,8179,3557,0.434894
4,19142,5976,0.312193
5,57328,20554,0.358533


In [65]:
has_comment = reviews_clean["review_comment_message"].notna()

In [66]:
print("Duplicate review_id:", reviews_clean["review_id"].duplicated().sum())
print("Duplicate order_id:", reviews_clean["order_id"].duplicated().sum())

Duplicate review_id: 814
Duplicate order_id: 551


In [67]:
duplicated_review_ids = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicated_review_ids.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [68]:
review_id_variation = (
    duplicated_review_ids
    .groupby("review_id")
    .agg(
        orders=("order_id", "nunique"),
        scores=("review_score", "nunique"),
        comments=("review_comment_message", "nunique"),
        creation_dates=("review_creation_date", "nunique"),
        answer_dates=("review_answer_timestamp", "nunique")
    )
)

review_id_variation.head(20)

,orders,scores,comments,creation_dates,answer_dates
review_id,,,,,
00130cbe1f9d422698c812ed8ded1919,2,1,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,1,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,1,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,1,0,1,1
0254bd905dc677a6078990aad3331a36,2,1,1,1,1
0288d42bef3dfe36930740c9588a570f,2,1,0,1,1
02aa7f5f75e964e3c7efa59a1f515281,2,1,0,1,1
034528cb00b6bc981847acafbdf2ae0b,2,1,0,1,1
03a6a25db577d0689440933055111897,2,1,1,1,1


In [69]:
id="review_exact_duplicates"
print(
    "Exact duplicate review rows:",
    reviews_clean.duplicated().sum()
)

Exact duplicate review rows: 0


In [70]:
order_items_clean = clean_datasets["order_items"]

print("Rows:", len(order_items_clean))
print("Unique order_id:", order_items_clean["order_id"].nunique())
print("Unique product_id:", order_items_clean["product_id"].nunique())
print("Unique seller_id:", order_items_clean["seller_id"].nunique())
print("Duplicate rows:", order_items_clean.duplicated().sum())

Rows: 112650
Unique order_id: 98666
Unique product_id: 32951
Unique seller_id: 3095
Duplicate rows: 0


In [71]:
orders_without_items = set(orders["order_id"]) - set(order_items_clean["order_id"])

print(f"Orders without items: {len(orders_without_items)}")

Orders without items: 775


In [72]:
orders_without_items_df = orders[
    orders["order_id"].isin(orders_without_items)
]

orders_without_items_df["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [73]:
order_items_clean[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [74]:
id="order_item_key"
print(
    "Duplicate order_id + order_item_id:",
    order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Duplicate order_id + order_item_id: 0


In [75]:
missing_product_refs = set(order_items_clean["product_id"]) - set(products_clean["product_id"])

print(f"Product IDs in order_items but missing from products: {len(missing_product_refs)}")

Product IDs in order_items but missing from products: 0


In [77]:
sellers_clean = clean_datasets["sellers"]

print(sellers_clean.shape)

(3095, 4)


In [78]:
missing_seller_refs = set(order_items_clean["seller_id"]) - set(sellers_clean["seller_id"])

print(f"Seller IDs in order_items but missing from sellers: {len(missing_seller_refs)}")

Seller IDs in order_items but missing from sellers: 0


In [79]:
id="products_keys"
products_clean = clean_datasets["products"]

print("Rows:", len(products_clean))
print("Unique product_id:", products_clean["product_id"].nunique())
print("Missing product_id:", products_clean["product_id"].isna().sum())
print("Exact duplicate rows:", products_clean.duplicated().sum())

Rows: 32951
Unique product_id: 32951
Missing product_id: 0
Exact duplicate rows: 0


In [80]:
category_counts = products_clean["product_category_name"].isna().value_counts()

category_counts

product_category_name
False    32341
True       610
Name: count, dtype: int64

In [81]:
category_translation_clean = clean_datasets["product_category_name_translation"]

print("Rows:", len(category_translation_clean))
print(
    "Unique category names:",
    category_translation_clean["product_category_name"].nunique()
)
print(
    "Duplicate rows:",
    category_translation_clean.duplicated().sum()
)
print(
    "Duplicate category names:",
    category_translation_clean["product_category_name"].duplicated().sum()
)

Rows: 71
Unique category names: 71
Duplicate rows: 0
Duplicate category names: 0


In [82]:
product_categories = set(
    products_clean["product_category_name"].dropna()
)

translated_categories = set(
    category_translation_clean["product_category_name"]
)

missing_translations = product_categories - translated_categories

print(
    f"Product categories without an English translation: "
    f"{len(missing_translations)}"
)

missing_translations

Product categories without an English translation: 2


{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

In [83]:
print("Rows:", len(sellers_clean))
print("Unique seller_id:", sellers_clean["seller_id"].nunique())
print("Missing seller_id:", sellers_clean["seller_id"].isna().sum())
print("Exact duplicate rows:", sellers_clean.duplicated().sum())

Rows: 3095
Unique seller_id: 3095
Missing seller_id: 0
Exact duplicate rows: 0


In [84]:
sellers_clean.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [85]:
customers_clean = clean_datasets["customers"]

print("Rows:", len(customers_clean))
print("Unique customer_id:", customers_clean["customer_id"].nunique())
print("Unique customer_unique_id:", customers_clean["customer_unique_id"].nunique())
print("Missing values:")
print(customers_clean.isna().sum())
print("Exact duplicate rows:", customers_clean.duplicated().sum())

Rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096
Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Exact duplicate rows: 0


In [86]:
missing_customer_refs = (
    set(orders["customer_id"])
    - set(customers_clean["customer_id"])
)

print(
    f"Customer IDs in orders but missing from customers: "
    f"{len(missing_customer_refs)}"
)

Customer IDs in orders but missing from customers: 0


In [87]:
for name, df in clean_datasets.items():
    print(f"{name:25} {df.shape}")

customers                 (99441, 5)
geolocation               (738332, 5)
orders                    (99441, 8)
order_items               (112650, 7)
order_payments            (103886, 5)
order_reviews             (99224, 7)
products                  (32951, 9)
sellers                   (3095, 4)
product_category_name_translation (71, 2)


In [88]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, df in clean_datasets.items():
    output_path = PROCESSED_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [89]:
list(PROCESSED_DIR.glob("*.csv"))

[WindowsPath('../data/processed/customers.csv'),
 WindowsPath('../data/processed/geolocation.csv'),
 WindowsPath('../data/processed/orders.csv'),
 WindowsPath('../data/processed/order_items.csv'),
 WindowsPath('../data/processed/order_payments.csv'),
 WindowsPath('../data/processed/order_reviews.csv'),
 WindowsPath('../data/processed/products.csv'),
 WindowsPath('../data/processed/product_category_name_translation.csv'),
 WindowsPath('../data/processed/sellers.csv')]

In [90]:
for name, df in clean_datasets.items():
    processed_path = PROCESSED_DIR / f"{name}.csv"
    processed_df = pd.read_csv(processed_path)

    print(
        f"{name:35} "
        f"memory={df.shape} | "
        f"saved={processed_df.shape}"
    )

customers                           memory=(99441, 5) | saved=(99441, 5)
geolocation                         memory=(738332, 5) | saved=(738332, 5)
orders                              memory=(99441, 8) | saved=(99441, 8)
order_items                         memory=(112650, 7) | saved=(112650, 7)
order_payments                      memory=(103886, 5) | saved=(103886, 5)
order_reviews                       memory=(99224, 7) | saved=(99224, 7)
products                            memory=(32951, 9) | saved=(32951, 9)
sellers                             memory=(3095, 4) | saved=(3095, 4)
product_category_name_translation   memory=(71, 2) | saved=(71, 2)


In [91]:
geo_processed = pd.read_csv(PROCESSED_DIR / "geolocation.csv")

print("Geolocation rows:", len(geo_processed))
print("Exact duplicate rows:", geo_processed.duplicated().sum())

Geolocation rows: 738332
Exact duplicate rows: 0


In [3]:
import pandas as pd

# Load the current processed products file
products_clean = pd.read_csv("../data/processed/products.csv")

# Columns that should contain integer values
product_integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# Convert them to nullable integers
for column in product_integer_columns:
    products_clean[column] = products_clean[column].astype("Int64")

# Save the corrected file
products_clean.to_csv(
    "../data/processed/products.csv",
    index=False
)

print("Products CSV updated successfully.")

Products CSV updated successfully.


In [5]:
products_clean[
    [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].dtypes

product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
dtype: object